In [7]:
import numpy as np
import pandas as pd
from PIL import Image, ImageFilter, ImageEnhance
import os
from sklearn.preprocessing import PowerTransformer
from sklearn.preprocessing import RobustScaler, MinMaxScaler, StandardScaler
import random
from concurrent.futures import ThreadPoolExecutor


import matplotlib.pyplot as plt
import seaborn as sns

## chuyển đổi hình ảnh trên tập dữ liệu CIC DDOS 2019

In [8]:
selected_columns = ['Protocol', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 
                    'Fwd Packets Length Total', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 
                    'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 
                    'Bwd Packet Length Mean', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 
                    'Flow IAT Max', 'Flow IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Min', 
                    'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 
                    'Bwd Header Length', 'Bwd Packets/s', 'Packet Length Max', 'FIN Flag Count', 'SYN Flag Count', 
                    'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'CWE Flag Count', 'ECE Flag Count', 'Down/Up Ratio', 
                    'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 
                    'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate', 'Init Fwd Win Bytes', 'Init Bwd Win Bytes', 
                    'Fwd Seg Size Min', 'Active Mean', 'Active Std', 'Active Max', 'Active Min', 'Idle Std', 'Label']

len(selected_columns)

50

In [9]:
df = pd.read_csv('data/csv/cicddos_2019_6_labels.csv')
df = df[selected_columns]

In [10]:

datadir = 'temp/data'

def convert2(df_normalized_splited, label, num):
    width = 7
    height = 7
    size = 224
    upscale_factor_with = size // width 
    upscale_factor_height = size // height

    new_image_size = (width * upscale_factor_with, height * upscale_factor_height)

    os.makedirs(f"{datadir}/{label}", exist_ok=True)  # Tạo thư mục nếu chưa có

    i = 1
    for row in df_normalized_splited.values:
        # Chuyển đổi dòng thành ma trận ảnh ban đầu
        image_array = np.array(row).reshape((width, height))
        image_array = np.nan_to_num(image_array)  # Thay thế giá trị NaN bằng 0

        # Phóng to mỗi pixel np.kron()
        upscale_matrix = np.ones((upscale_factor_with, upscale_factor_height))
        enlarged_image_array = np.kron(image_array, upscale_matrix)

        # Chuyển đổi sang ảnh
        image = Image.fromarray((enlarged_image_array * 255).astype(np.uint8))  # Chuyển sang RGB
        image = image.convert("RGB")

        # ###### thêm nhiễu vào ảnh ######
        # rotate_prob = random.random() < 0.4 # xoay
        # flip_prob = random.random() < 0.4 # lật
        # blur_prob = random.random() <= 0.4  # xác suất làm mờ
        # noise_prob = random.random() <= 0.4  # xác suất thêm nhiễu

        # # Xoay ảnh ngẫu nhiên
        # if rotate_prob:  
        #     angle = random.uniform(-90, 90)  
        #     image = image.rotate(angle)

        # # Lật ảnh ngẫu nhiên
        # if flip_prob:
        #     if random.random() < 0.5:
        #         image = image.transpose(Image.FLIP_LEFT_RIGHT)  # Lật ngang
        #     else:
        #         image = image.transpose(Image.FLIP_TOP_BOTTOM)  # Lật dọc

        # # làm mờ ảnh
        # if blur_prob:
        #     image = image.filter(ImageFilter.GaussianBlur(radius=random.uniform(10, 20))) # mức độ mờ

        # #làm nhiễu ảnh
        # if noise_prob:
        #     noise = np.random.normal(25, 50, (new_image_size[0], new_image_size[1]))  # Thêm nhiễu Gaussian
        #     noisy_image_array = np.array(image.convert("L")) + noise  # Chuyển sang grayscale trước khi thêm nhiễu
        #     noisy_image_array = np.clip(noisy_image_array, 0, 255).astype(np.uint8)  # Giữ giá trị trong khoảng 0-255
        #     image = Image.fromarray(noisy_image_array).convert("RGB")

        # Lưu ảnh
        image = image.resize((size, size))
        image.save(f"{datadir}/{label}/{str(i)}.png")
        

        i += 1
        if i > num:
            break

def convert(df_normalized_splited, label, num):
    width = 7
    height = 7
    size = 224
    upscale_factor_with = size // width 
    upscale_factor_height = size // height

    new_image_size = (width * upscale_factor_with, height * upscale_factor_height)

    os.makedirs(f"{datadir}/{label}", exist_ok=True)  # Tạo thư mục nếu chưa có

    i = 1
    for row in df_normalized_splited.values:
        # Chuyển đổi dòng thành ma trận ảnh ban đầu
        image_array = np.array(row).reshape((width, height))
        image_array = np.nan_to_num(image_array)  # Thay thế giá trị NaN bằng 0

        # Phóng to mỗi pixel np.kron()
        upscale_matrix = np.ones((upscale_factor_with, upscale_factor_height))
        enlarged_image_array = np.kron(image_array, upscale_matrix)

         # Vẽ heatmap bằng seaborn
        plt.figure(figsize=(2, 2))  # Kích thước ảnh
        sns.heatmap(enlarged_image_array, cmap="jet", cbar=False)  # Chọn colormap "jet" để có màu sắc đẹp
        plt.axis("off")  # Ẩn trục
        # Lưu ảnh
        plt.savefig(f"{datadir}/{label}/{str(i)}.png", bbox_inches='tight', pad_inches=0)
        plt.close()

        i += 1
        if i > num:
            break

def setup_to_convert(df_normalized, label):
    os.makedirs(f'data/{datadir}/{label}', exist_ok=True)

    # tổng số lượng ảnh train + valid + test của mỗi nhãn
    n = 500
    convert(df_normalized, label, num=n)

In [ ]:
df1 = df.drop(columns=['Label'])
df2 = df['Label']

df1.replace([-np.inf, np.inf], 0, inplace=True)
df1.fillna(df1.mean(), inplace=True)  # Điền giá trị NaN bằng giá trị trung vị

std = df1.std()
std.replace(0, 1, inplace=True)

df1 = (df1 - df1.mean()) / std

df1 = np.log1p(df1 + 1)
df1 = pd.DataFrame(MinMaxScaler(feature_range=(0, 255)).fit_transform(df1).astype(np.uint8))

grouped =  pd.concat([df1, df2], ignore_index=True)
# Nhóm dữ liệu theo cột 'Label'
grouped = df.groupby('Label')

# Tạo dictionary để lưu các DataFrame tương ứng với từng nhãn
dfs = {label: group for label, group in grouped}

def process_label(label, df_label):
    df_drop_label = df_label.drop(columns=['Label'])
    data_normalized = df_drop_label
    setup_to_convert(data_normalized, label)

# Sử dụng ThreadPoolExecutor để chạy đa luồng với tối đa 5 luồng
with ThreadPoolExecutor(max_workers=3) as executor:
    futures = [executor.submit(process_label, label, df_label) for label, df_label in dfs.items()]

    # Đợi tất cả các task hoàn thành
    for future in futures:
        future.result()

print("Hoàn thành xử lý đa luồng!")

f:\NCKH\code\DN\IDS-RESNET-50-System\train_model\.venv\Lib\site-packages\pandas\core\internals\blocks.py:393: RuntimeWarning: invalid value encountered in log1p
  result = func(self.values, **kwargs)
C:\Users\NewTun\AppData\Local\Temp\ipykernel_19032\2165434559.py:13: RuntimeWarning: invalid value encountered in cast
  df1 = pd.DataFrame(MinMaxScaler(feature_range=(0, 255)).fit_transform(df1).astype(np.uint8))
